In [ ]:
# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install pandas pyarrow openpyxl -q
# import apply_human_verified

In [ ]:
#!/usr/bin/env python3
"""
apply_human_verified.py  —  Google Colab
========================================
Freeze the HUMAN-VERIFIED corpus as the single denominator for every downstream
analysis, and score the screener against that ground truth.

TWO JOBS
  A) GOLD FILTER — read the human-reviewed spreadsheet, keep rows where
     `final_include` is non-empty, and subset EVERY derived parquet by oa_id.
     Enrichment is keyed on oa_id and cached, so NOTHING is re-harvested; we only
     re-scope. Re-run afterwards only what depends on corpus composition
     (citation clustering; BERTopic re-fit from cached embeddings).
  B) SCREENER VALIDATION — the same file contains predictions (llm_label,
     gpt_label, deepseek_label, match_vs_rule) next to truth (human/final_include),
     so it yields precision/recall/F1 BY REGION = Supp. Table S1 / data-paper T-V1.

INPUT: the reviewed .xlsx (Drive). Columns used:
  oa_id, final_include (non-empty = INCLUDE), human, llm_label, gpt_label,
  deepseek_label, match_vs_rule, oa_relevance_tier
SETUP
  !pip install pandas pyarrow openpyxl -q
"""

import re, sys
from pathlib import Path
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
REVIEWED_XLSX = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen_human_validation.xlsx"
BASE_DIR      = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data"
GOLD_IDS      = BASE_DIR + "/gold_included_ids.parquet"
VALID_OUT     = BASE_DIR + "/screener_validation_by_region.csv"
SUFFIX        = "_gold"          # filtered copies get this suffix

# every derived parquet to re-scope (skip any that don't exist)
DERIVED = [
    "screened_llm.parquet", "with_study_location.parquet", "with_methods_triples.parquet",
    "funding_enrich.parquet", "ror_enrich.parquet", "with_venue.parquet", "with_doi.parquet",
    "with_gender.parquet", "with_capacity.parquet", "with_burden.parquet",
    "with_risk.parquet", "with_livestock.parquet", "with_clusters.parquet",
    "with_topics.parquet", "with_upw_oc.parquet", "with_pmcid.parquet", "with_orcid.parquet",
    "with_methods_triples.parquet", "citation_clusters.parquet", "screened_llm.parquet"
]

MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
       "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
       "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x):
    return S(x).rsplit("/", 1)[-1]
def truthy(x):
    """final_include: non-empty = include. Handles 1/yes/true/include/x."""
    v = S(x).strip().lower()
    return v not in ("", "0", "no", "false", "exclude", "n", "nan", "none")


# =========================================================================== #
# A) GOLD FILTER                                                              #
# =========================================================================== #
def load_reviewed():
    rv = pd.read_excel(REVIEWED_XLSX, dtype=str)
    rv.columns = [c.strip() for c in rv.columns]
    if "final_include" not in rv.columns:
        raise SystemExit("[!] no `final_include` column found")
    rv["oa_id"] = rv["oa_id"].map(short_id)
    rv["_include"] = rv["final_include"].map(truthy)
    n_in = int(rv["_include"].sum())
    print(f"[reviewed] {len(rv):,} rows reviewed | INCLUDE {n_in:,} "
          f"({n_in/max(len(rv),1):.1%}) | EXCLUDE {len(rv)-n_in:,}", file=sys.stderr)
    return rv

def write_gold_ids(rv):
    gold = rv.loc[rv["_include"], ["oa_id"]].drop_duplicates()
    gold.to_parquet(GOLD_IDS, index=False)
    print(f"[write] {GOLD_IDS}  ({len(gold):,} included ids)", file=sys.stderr)
    return set(gold["oa_id"])

def subset_derived(gold_ids):
    print("\n===== RE-SCOPING DERIVED PARQUETS (no re-harvesting) =====")
    print(f"{'file':<38}{'before':>10}{'after':>10}{'kept':>8}")
    for name in DERIVED:
        p = Path(BASE_DIR) / name
        if not p.exists():
            print(f"{name:<38}{'--':>10}{'--':>10}{'skip':>8}"); continue
        try:
            df = pd.read_parquet(p)
        except Exception as e:
            print(f"{name:<38}   read error: {e}"); continue
        if "oa_id" not in df.columns:
            print(f"{name:<38}   no oa_id column — skipped"); continue
        before = len(df)
        df["_sid"] = df["oa_id"].map(short_id)
        out = df[df["_sid"].isin(gold_ids)].drop(columns=["_sid"])
        outp = p.with_name(p.stem + SUFFIX + p.suffix)
        out.to_parquet(outp, index=False)
        print(f"{name:<38}{before:>10,}{len(out):>10,}{len(out)/max(before,1):>7.1%}")
    print(f"\n[note] filtered copies written with suffix '{SUFFIX}'. Point all "
          f"downstream scripts at these.", file=sys.stderr)


# =========================================================================== #
# B) SCREENER VALIDATION BY REGION  (Supp. Table S1 / data-paper T-V1)         #
# =========================================================================== #
def dominant_bloc(sb):
    parts = {x for x in re.split(r"[;|,]", S(sb)) if x}
    if parts & SSA or "ssa" in S(sb).lower(): return "ssa"
    if parts & MAGHREB or "maghreb" in S(sb).lower(): return "maghreb"
    return "other"

def pred_positive(x):
    v = S(x).strip().lower()
    return v in ("1", "true", "yes", "relevant", "include", "tick_relevant", "y")

def score_screener(rv, bloc_map=None):
    """Truth = final_include. Predictions = each screener column present."""
    rv = rv.copy()
    rv["_bloc"] = (rv["oa_id"].map(bloc_map).fillna("other")
                   if bloc_map else "other")
    rows = []
    preds = [c for c in ("llm_label", "gpt_label", "deepseek_label", "match_vs_rule",
                         "oa_relevance_tier") if c in rv.columns]
    for col in preds:
        for bloc, g in list(rv.groupby("_bloc")) + [("ALL", rv)]:
            y_true = g["_include"].astype(int)
            y_pred = g[col].map(pred_positive).astype(int)
            tp = int(((y_pred == 1) & (y_true == 1)).sum())
            fp = int(((y_pred == 1) & (y_true == 0)).sum())
            fn = int(((y_pred == 0) & (y_true == 1)).sum())
            tn = int(((y_pred == 0) & (y_true == 0)).sum())
            prec = tp / (tp + fp) if tp + fp else float("nan")
            rec = tp / (tp + fn) if tp + fn else float("nan")
            f1 = 2 * prec * rec / (prec + rec) if prec and rec and (prec + rec) else float("nan")
            rows.append({"screener": col, "bloc": bloc, "n": len(g),
                         "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                         "precision": prec, "recall": rec, "f1": f1})
    tab = pd.DataFrame(rows)
    tab.to_csv(VALID_OUT, index=False)
    print("\n===== SCREENER PERFORMANCE vs HUMAN GROUND TRUTH =====")
    show = tab.copy()
    for c in ("precision", "recall", "f1"):
        show[c] = show[c].map(lambda v: f"{v:.3f}" if pd.notna(v) else "  -  ")
    print(show[["screener", "bloc", "n", "tp", "fp", "fn", "precision", "recall", "f1"]]
          .to_string(index=False))
    print(f"\n[write] {VALID_OUT}", file=sys.stderr)
    print("[use] this is Supp. Table S1 (Paper 3) and Technical Validation T-V1 "
          "(data paper). Report precision/recall PER BLOC.", file=sys.stderr)
    return tab


# =========================================================================== #
if __name__ == "__main__":
    rv = load_reviewed()
    gold_ids = write_gold_ids(rv)

    # bloc labels for the by-region validation: take them from the study-location
    # parquet if it exists (reviewed sheet has no bloc column)
    bloc_map = None
    slp = Path(BASE_DIR) / "with_study_location.parquet"
    if slp.exists():
        try:
            sl = pd.read_parquet(slp, columns=["oa_id", "study_bloc"])
            sl["oa_id"] = sl["oa_id"].map(short_id)
            bloc_map = dict(zip(sl["oa_id"], sl["study_bloc"].map(dominant_bloc)))
            print(f"[bloc] labels for {len(bloc_map):,} ids from study-location",
                  file=sys.stderr)
        except Exception as e:
            print(f"[bloc] could not load study_bloc ({e}); scoring ALL only",
                  file=sys.stderr)

    score_screener(rv, bloc_map)
    subset_derived(gold_ids)

    print("\n===== NEXT =====")
    print("1. Re-run citation clustering on with_livestock_gold.parquet "
          "(coupling graph depends on corpus composition).")
    print("2. Re-fit BERTopic on the gold subset — embeddings are cached, so it "
          "only re-clusters (seconds).")
    print("3. Re-run equity metrics / JSD / misalignment on the *_gold files.")
    print("4. Enrichment does NOT need re-running: it is keyed on oa_id and cached.")